# Onion Quality Assessment — End-to-End Colab Training & Local Export
**Handheld Onion Quality Assessment & Multi-Standard Grading (DoCA + AGMARK)**

This workbook runs in **Google Colab via the VS Code extension**.
It executes the complete Phase 2 model training pipeline:
- **Model 1**: YOLOv8-seg single-class `onion` detector & segmenter (Target: mAP@0.5 ≥ 85% on D1-B benchmark)
- **Model 2**: 4-class defect CNN on bulb crops (`healthy`, `sprouted`, `rotten`, `mechanical_damage`) (Target: F1 ≥ 90%)
- **Export**: Lightweight ONNX models for handheld mobile inference
- **Local Download**: Programmatic transfer cell allowing instant download to your local PC without using Colab's web UI.

### Step 1: Environment & GPU Verification

In [ ]:
!nvidia-smi
!pip install -q "ultralytics>=8.3.0" "roboflow>=1.1.0" "opencv-python-headless>=4.10.0" "pydantic>=2.8.0" "imagehash>=4.3.1" pandas matplotlib tqdm pyyaml

### Step 2: Download Public Datasets via Roboflow API
Enter your free Roboflow API key below to download D1-A, D1-B, and D2-A directly onto Colab's high-speed cloud disk.

In [ ]:
import os
from roboflow import Roboflow

# Enter your Roboflow Private API Key here:
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "YOUR_API_KEY_HERE")

os.makedirs("/content/data/raw", exist_ok=True)
os.makedirs("/content/runs", exist_ok=True)
os.makedirs("/content/exports", exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

print("1. Downloading D1-A (YOLOv8 instance segmentation)... ")
d1a_project = rf.workspace("yolo-custom-object-detection").project("instance-segmentation-wagk9")
d1a_data = d1a_project.version(12).download("yolov8", location="/content/data/raw/D1A_instance_seg")

print("\n2. Downloading D1-B (onionthesis1 - cross-source test set)... ")
d1b_project = rf.workspace("paul-angelo-lavarias-ii").project("onionthesis1")
d1b_data = d1b_project.version(1).download("yolov8", location="/content/data/raw/D1B_onionthesis1")

print("\n3. Downloading D2-A (onion-disease - defect classifier source)... ")
d2a_project = rf.workspace("anas-kadiri-ffnxr").project("onion-disease")
d2a_data = d2a_project.version(1).download("yolov8", location="/content/data/raw/D2A_onion_disease")

print("\n All datasets downloaded successfully into /content/data/raw/")

### Step 3: Prepare Model 1 Dataset (Single-Class 'onion')
Unifies all masks to class `0: onion`, cleans negative labels from D1-B, and generates `data.yaml`.

In [ ]:
import shutil
from pathlib import Path
import yaml

def prepare_model1_split(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, target_classes=[0]):
    Path(dst_img_dir).mkdir(parents=True, exist_ok=True)
    Path(dst_lbl_dir).mkdir(parents=True, exist_ok=True)
    
    count = 0
    for img_p in Path(src_img_dir).glob("*.jpg"):
        lbl_p = Path(src_lbl_dir) / f"{img_p.stem}.txt"
        dest_img = Path(dst_img_dir) / img_p.name
        dest_lbl = Path(dst_lbl_dir) / f"{img_p.stem}.txt"
        
        shutil.copy2(img_p, dest_img)
        if not lbl_p.exists():
            dest_lbl.touch()
            continue
            
        with open(lbl_p, "r") as f:
            lines = f.readlines()
        valid = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 6 and int(parts[0]) in target_classes:
                valid.append("0 " + " ".join(parts[1:]) + "\n")
        with open(dest_lbl, "w") as f:
            f.writelines(valid)
        count += 1
    return count

print("Formatting Model 1 dataset...")
prepare_model1_split("/content/data/raw/D1A_instance_seg/train/images", "/content/data/raw/D1A_instance_seg/train/labels", "/content/data/model1/train/images", "/content/data/model1/train/labels")
prepare_model1_split("/content/data/raw/D1A_instance_seg/valid/images", "/content/data/raw/D1A_instance_seg/valid/labels", "/content/data/model1/val/images", "/content/data/model1/val/labels")
prepare_model1_split("/content/data/raw/D1B_onionthesis1/train/images", "/content/data/raw/D1B_onionthesis1/train/labels", "/content/data/model1/test/images", "/content/data/model1/test/labels", target_classes=[0])

data_yaml = {
    "path": "/content/data/model1",
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {0: "onion"}
}
with open("/content/data/model1/data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

print("\n Model 1 data.yaml ready at /content/data/model1/data.yaml")

### Step 4: Train Model 1 (YOLOv8n-seg)
Target: **mAP@0.5 ≥ 85%** on cross-source benchmark.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8-nano segmentation weights
model1 = YOLO("yolov8n-seg.pt")

results_m1 = model1.train(
    data="/content/data/model1/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    degrees=15.0,     # Rotation augmentation for handheld views
    fliplr=0.5,
    flipud=0.5,
    mosaic=1.0,
    project="/content/runs/model1",
    name="yolov8_seg_onion",
    save=True,
    plots=True
)

### Step 5: Evaluate Model 1 on D1-B Cross-Source Benchmark
Tests the model on an independent author's dataset (unseen background and lighting).

In [ ]:
# Benchmark on unseen D1-B test split
metrics_m1 = model1.val(data="/content/data/model1/data.yaml", split="test")
print(f"\n" + "="*60)
print(f"Model 1 D1-B Benchmark mAP@0.5: {metrics_m1.box.map50:.4f}")
print(f"Model 1 Mask mAP@0.5:           {metrics_m1.seg.map50:.4f}")
print("="*60)

### Step 6: Prepare Model 2 Defect Crops (4 Core Classes)
Extracts individual bulb crops from D2-A and categorizes them into `healthy`, `sprouted`, `rotten`, `mechanical_damage`.

In [ ]:
import cv2
import random

CLASS_MAP = {
    "onion": "healthy", "healthy": "healthy",
    "rotten": "rotten", "sprouted-rotten": "rotten",
    "sprouted": "sprouted", "sprouted-cuts": "sprouted", "sprouted-discoloured": "sprouted",
    "sprouted-blacksmutinfected": "sprouted", "sprouted-rooting": "sprouted", "rooted": "sprouted", "rooting": "sprouted",
    "cuts": "mechanical_damage", "blackmutinfected-cuts": "mechanical_damage"
}

classes = ["healthy", "sprouted", "rotten", "mechanical_damage"]
for s in ["train", "val", "test"]:
    for c in classes:
        Path(f"/content/data/model2/{s}/{c}").mkdir(parents=True, exist_ok=True)

# Read data.yaml from D2A to get class names
with open("/content/data/raw/D2A_onion_disease/data.yaml") as f:
    d2a_cfg = yaml.safe_load(f)
raw_names = d2a_cfg.get("names", [])
if isinstance(raw_names, dict):
    raw_names = [raw_names[k] for k in sorted(raw_names.keys())]

crops_by_class = {c: [] for c in classes}
for img_p in Path("/content/data/raw/D2A_onion_disease").rglob("*.jpg"):
    lbl_p = img_p.with_suffix(".txt")
    if not lbl_p.exists(): continue
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    
    with open(lbl_p) as f: lines = f.readlines()
    for idx, l in enumerate(lines):
        p = l.strip().split()
        if len(p) < 5: continue
        cid = int(p[0])
        cname = raw_names[cid] if cid < len(raw_names) else str(cid)
        target_cls = CLASS_MAP.get(cname)
        if not target_cls: continue
        
        cx, cy, bw, bh = float(p[1]), float(p[2]), float(p[3]), float(p[4])
        x1, y1 = max(0, int((cx - bw*0.54)*w)), max(0, int((cy - bh*0.54)*h))
        x2, y2 = min(w, int((cx + bw*0.54)*w)), min(h, int((cy + bh*0.54)*h))
        if x2 > x1 and y2 > y1:
            crop = cv2.resize(img[y1:y2, x1:x2], (224, 224))
            crops_by_class[target_cls].append((crop, f"{img_p.stem}_c{idx}.jpg"))

# Save train (70%), val (15%), test (15%)
for c, items in crops_by_class.items():
    random.shuffle(items)
    n1 = int(len(items)*0.70)
    n2 = int(len(items)*0.85)
    for crop, name in items[:n1]: cv2.imwrite(f"/content/data/model2/train/{c}/{name}", crop)
    for crop, name in items[n1:n2]: cv2.imwrite(f"/content/data/model2/val/{c}/{name}", crop)
    for crop, name in items[n2:]: cv2.imwrite(f"/content/data/model2/test/{c}/{name}", crop)
    print(f"  - {c}: {len(items)} crops")

print("\n Model 2 ImageFolder dataset ready at /content/data/model2")

### Step 7: Train Model 2 (Defect Classifier)
Target: **F1 ≥ 90%** on defect classes.

In [ ]:
model2 = YOLO("yolov8n-cls.pt")

results_m2 = model2.train(
    data="/content/data/model2",
    epochs=40,
    imgsz=224,
    batch=32,
    project="/content/runs/model2",
    name="defect_classifier_4class",
    save=True,
    plots=True
)

metrics_m2 = model2.val(data="/content/data/model2", split="test")
print(f"\nModel 2 Top-1 Test Accuracy: {metrics_m2.top1:.4f}")

### Step 8: Export Both Models to ONNX for Mobile Deployment

In [ ]:
# Export Model 1 (detection + segmentation) with dynamic shape for mobile resolution flexibility
onnx_m1 = model1.export(format="onnx", dynamic=True)
# Export Model 2 (defect classifier) at fixed 224x224
onnx_m2 = model2.export(format="onnx", imgsz=224)

print(f"\n Exported Model 1 ONNX: {onnx_m1}")
print(f" Exported Model 2 ONNX: {onnx_m2}")

### Step 9: Download Trained Models Directly to Your PC (No Colab Web UI Needed)
Run the cell below. It packages all `.pt` weights, `.onnx` models, and evaluation plots, uploads them, and gives you an exact 1-line `curl` command to run in your local PC terminal.

In [ ]:
import subprocess

# 1. Organize exports
!mkdir -p /content/exports/weights /content/exports/onnx /content/exports/eval_plots
!cp /content/runs/model1/yolov8_seg_onion/weights/best.pt /content/exports/weights/model1_yolov8_seg.pt || true
!cp /content/runs/model2/defect_classifier_4class/weights/best.pt /content/exports/weights/model2_defect_cls.pt || true
!cp /content/runs/model1/yolov8_seg_onion/weights/best.onnx /content/exports/onnx/model1_yolov8_seg.onnx || true
!cp /content/runs/model2/defect_classifier_4class/weights/best.onnx /content/exports/onnx/model2_defect_cls.onnx || true
!cp /content/runs/model1/yolov8_seg_onion/*.png /content/exports/eval_plots/ || true
!cp /content/runs/model2/defect_classifier_4class/*.png /content/exports/eval_plots/ || true

# 2. Zip archive
archive_path = "/content/onion_model_artifacts.zip"
!zip -r {archive_path} /content/exports

# 3. Programmatic CLI Download via transfer.sh
print("\nUploading package to transfer.sh for direct terminal download...")
res = subprocess.run(f"curl --upload-file {archive_path} https://transfer.sh/onion_model_artifacts.zip", shell=True, capture_output=True, text=True)
url = res.stdout.strip()

print("\n" + "="*80)
print(" TRAINING COMPLETE! DOWNLOAD ARTIFACTS TO YOUR PC WITH THIS COMMAND:")
print("="*80)
print(f"\nmkdir -p ~/Projects/Onion_grading_new/runs && curl -o ~/Projects/Onion_grading_new/runs/onion_model_artifacts.zip {url} && unzip -o ~/Projects/Onion_grading_new/runs/onion_model_artifacts.zip -d ~/Projects/Onion_grading_new/runs/\n")
print("="*80)

### (Optional Alternative) Google Drive Mount Backup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/onion_grading_runs
    !cp /content/onion_model_artifacts.zip /content/drive/MyDrive/onion_grading_runs/
    print("\n Saved backup to Google Drive at: /content/drive/MyDrive/onion_grading_runs/")
except Exception as e:
    print(f"Drive backup skipped: {e}")